# Статистический анализ точности системы лазерного наведения с БПЛА

В этой главе оценивается точность системы лазерного наведения, установленной на квадрокоптере со стабилизированным подвесом. Практический вопрос состоит в том, при каких условиях полёта лазерное пятно остаётся достаточно близко к заданной реперной точке и какие параметры полёта сильнее всего формируют ошибку.

Поскольку отклонение пятна имеет две координатные составляющие, для дальнейшего анализа нужен один итоговый показатель ошибки. Таким показателем является радиальная ошибка:

$$
R = \sqrt{r_x^2 + r_y^2}.
$$

Здесь $r_x$ и $r_y$ — отклонения пятна по двум горизонтальным осям, мм. Квадраты исключают взаимное погашение отклонений разных знаков, а корень возвращает показатель в исходную размерность. Иными словами, $R$ показывает полное расстояние от фактического положения пятна до заданной точки, поэтому далее именно $R$ используется как результативный признак.

Для единообразия обозначений в главе используются следующие величины:

| Обозначение | Смысл | Единицы |
|---|---|---|
| $R$ | радиальная ошибка наведения | мм |
| $H$ | высота зависания БПЛА | м |
| $V$ | локальная скорость ветра | м/с |
| $N_{\text{sat}}$ | число видимых спутников GNSS | шт. |
| $C$ | облачность | % |
| $\hat R$ | значение ошибки, рассчитанное регрессионной моделью | мм |

### Характеристика выборки

| Параметр | Значение |
|---|---|
| Период проведения испытаний | апрель — октябрь 2025 г. |
| Число лётных дней | 59 |
| Контрольные точки | 5 (P1–P5) |
| Высоты зависания | $H \in \{3,\; 5,\; 10,\; 15,\; 20\}$ м |
| Повторов на каждую комбинацию (точка × высота) | 3 |
| Общее число измерений | $59 \times 5 \times 5 \times 3 = 4425$ |

Для каждого измерения фиксировались время, координаты точки, высота $H$, номер повтора, скорость ветра $V$, порывы ветра, направление ветра, облачность $C$, число спутников $N_{\text{sat}}$, компоненты отклонения $r_x$, $r_y$ и итоговая радиальная ошибка $R$. Такая структура данных позволяет последовательно перейти от описательной статистики к дисперсионному анализу факторов и затем к регрессионному уравнению прогноза.

In [155]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm


def show_eq(model, response="R"):
    b = model.params
    parts = [f"{b['Intercept']:.3g}"]
    for name in ["H", "V", "H:V"]:
        if name in b:
            sym = {"H": "H", "V": "V", "H:V": r"H\!\cdot\!V"}[name]
            parts.append(f"{b[name]:+.3g}\\,{sym}")
    eq = " ".join(parts).replace("+ -", "- ")
    display(Markdown(rf"$$\hat{{{response}}} = {eq} \quad (R^2 = {model.rsquared:.3f})$$"))


BASE_DIR = Path(".").resolve()
OUTPUT_DIR = BASE_DIR / "output"
JSON_PATH = BASE_DIR / "balanced_selection" / "selected_days_2024-2025.json"

H_ORDER = [3, 5, 10, 15, 20]
H_CAT_ORDER = ["3 м", "5 м", "10 м", "15 м", "20 м"]
ALPHA = 0.05

COL_MAP = {
    "Время": "time", "Точка": "point", "H (м)": "H", "Полёт": "flight",
    "V (м/с)": "V", "Порывы, откр. источник (м/с)": "gusts",
    "Направление ветра": "wind_dir", "Облачность (%)": "cloud",
    "Спутники": "sat", "r_x (мм)": "r_x", "r_y (мм)": "r_y", "R (мм)": "R"
}


def load_measurements(input_dir: Path) -> pd.DataFrame:
    frames = []
    for fpath in sorted(input_dir.glob("*.xlsx")):
        if fpath.name.startswith("~$"):
            continue
        tmp = pd.read_excel(fpath, header=12, engine="openpyxl")
        tmp.rename(columns=COL_MAP, inplace=True)
        tmp["date"] = fpath.stem
        frames.append(tmp)
    df = pd.concat(frames, ignore_index=True)
    for c in ("H", "V", "cloud", "sat", "R", "r_x", "r_y", "gusts", "wind_dir"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df.dropna(subset=["H", "V", "R"], inplace=True)
    return df


weather = pd.DataFrame(json.loads(JSON_PATH.read_text(encoding="utf-8")))
weather["date"] = pd.to_datetime(weather["date"])

df = load_measurements(OUTPUT_DIR)
df["H_cat"] = df["H"].astype(int).astype(str) + " м"
df["V_bin"] = pd.cut(
    df["V"],
    bins=[0, 2.2, 2.9, 3.6, 4.4, 100],
    labels=["0.8–2.2", "2.2–2.9", "2.9–3.6", "3.6–4.4", "4.4–7.8"],
)

print(f"Загружено: {len(df)} измерений за {df['date'].nunique()} дней")
print(f"Погодный JSON: {len(weather)} дней")
print(f"Высоты: {sorted(df['H'].unique())} м")
print(f"Точки: {', '.join(sorted(df['point'].unique()))}")
print(f"Повторов на высоту: {df['flight'].nunique()}")

ALL_FIGS: list[tuple[str, go.Figure]] = []


Загружено: 4425 измерений за 59 дней
Погодный JSON: 59 дней
Высоты: [np.int64(3), np.int64(5), np.int64(10), np.int64(15), np.int64(20)] м
Точки: P1, P2, P3, P4, P5
Повторов на высоту: 3


### Обзор выборки и условий эксперимента

Перед анализом ошибки необходимо проверить, в каких условиях получены измерения. Если все полёты выполнены в одном погодном режиме, влияние высоты $H$ и ветра $V$ может быть смешано с особенностями отдельных дней. Поэтому сначала рассматриваются состав погодных режимов, диапазоны метеопараметров и общее распределение радиальной ошибки $R$.

In [156]:
weather_counts = weather["weather_code_text"].value_counts().reset_index()
weather_counts.columns = ["Погода", "Дней"]

fig = px.bar(
    weather_counts, x="Погода", y="Дней",
    color="Погода", text="Дней",
    title="Рис. 1. Распределение лётных дней по типу погоды",
)
fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="Количество дней")
fig.update_traces(textposition="outside")
ALL_FIGS.append(("fig_01_weather_types", fig))
fig.show()

In [157]:
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    "Температура (°C)", "Средний ветер (м/с)",
    "Максимальные порывы (м/с)", "Облачность (%)"
])

for i, (col, name) in enumerate([
    ("temperature_2m_mean", "Температура"),
    ("wind_speed_10m_mean", "Ветер"),
    ("wind_gusts_10m_max", "Порывы"),
    ("cloud_cover_mean", "Облачность"),
]):
    r, c = divmod(i, 2)
    fig.add_trace(
        go.Histogram(x=weather[col], nbinsx=15, name=name, showlegend=False),
        row=r + 1, col=c + 1
    )

fig.update_layout(height=500, title_text="Рис. 2. Распределение метеопараметров по дням выборки")
ALL_FIGS.append(("fig_02_meteo_histograms", fig))
fig.show()

In [158]:
fig = px.histogram(
    df, x="R", nbins=80, marginal="box",
    title="Рис. 3. Общее распределение радиальной ошибки R",
    labels={"R": "R (мм)"},
    color_discrete_sequence=["steelblue"],
)
fig.update_layout(yaxis_title="Количество измерений", height=400)
ALL_FIGS.append(("fig_03_R_distribution", fig))
fig.show()

Рис. 1 показывает, что выборка не сведена к одному погодному сценарию: кроме 27 пасмурных дней, в неё вошли дни с моросью разной интенсивности, ясная погода и переменная облачность. Для исследования это важно, потому что точность наведения оценивается не только в спокойных, но и в неблагоприятных полевых условиях.

На рис. 2 метеопараметры покрывают рабочий диапазон испытаний: средний ветер изменяется примерно от 1 до 7 м/с, максимальные порывы — примерно от 4 до 18 м/с, облачность — от почти нулевой до 100%. Следовательно, в выборке присутствуют как мягкие, так и напряжённые режимы полёта, а фактор ветра можно проверять не формально, а по реально наблюдавшемуся диапазону.

Рис. 3 показывает правоасимметричное распределение $R$: основная масса измерений сосредоточена примерно в диапазоне 50–250 мм, но правый хвост уходит за 1000 мм. Иными словами, типичная ошибка умеренная, однако при неблагоприятных сочетаниях высоты и ветра возникают крупные промахи. Этот факт заранее задаёт ограничение для дальнейших моделей: линейное уравнение будет описывать основной тренд, но редкие хвостовые значения потребуют осторожной интерпретации.

### Описательная статистика

После общего обзора выборки перейдём от отдельных измерений к компактным характеристикам ошибки. Для каждой высоты $H$ сначала вычисляется среднее значение радиальной ошибки:

$$
\bar{R} = \frac{1}{n}\sum_{i=1}^{n} R_i.
$$

Здесь $R_i$ — ошибка в отдельном измерении, а $n$ — число измерений в высотной группе. Суммирование объединяет все наблюдения данной высоты, а деление на $n$ переводит сумму в средний уровень. Поэтому $\bar R$ используется как характеристика типичной ошибки на фиксированной высоте.

Одного среднего значения недостаточно: две высоты могут иметь близкую среднюю ошибку, но разную устойчивость результата. Поэтому дополнительно рассчитывается выборочное стандартное отклонение:

$$
s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(R_i - \bar{R})^2}.
$$

Разность $R_i - \bar R$ показывает отклонение отдельного измерения от среднего уровня, квадрат устраняет знак отклонения, а корень возвращает показатель в миллиметры. Иными словами, $s$ показывает разброс повторных измерений вокруг типичной ошибки. Если вместе со средним растёт и $s$, то ухудшается не только точность, но и повторяемость наведения.

In [159]:
desc = (
    df.groupby("H", sort=True)["R"]
    .agg(
        N="count",
        Среднее="mean",
        Ст_откл="std",
        Минимум="min",
        Максимум="max",
        P95=lambda x: x.quantile(0.95)
    )
    .round(1)
)
desc.index.name = "H (м)"
desc.columns = ["N", "Среднее, мм", "σ, мм", "Мин, мм", "Макс, мм", "R₉₅, мм"]

fig = go.Figure(data=[go.Table(
    columnwidth=[50, 50, 80, 70, 60, 70, 70],
    header=dict(
        values=["<b>H (м)</b>"] + [f"<b>{c}</b>" for c in desc.columns],
        fill_color="#34495e", font=dict(color="white", size=13),
        align="center", height=32,
    ),
    cells=dict(
        values=[desc.index.tolist()] + [desc[c].tolist() for c in desc.columns],
        fill_color=[["#ecf0f1", "#ffffff"] * 3],
        font=dict(size=12), align="center", height=28,
    )
)])
fig.update_layout(
    title="Таблица 1. Описательная статистика радиальной ошибки R по высотам",
    width=820, height=260, margin=dict(l=10, r=10, t=50, b=10),
    font=dict(family="Arial"),
)
ALL_FIGS.append(("fig_desc_table", fig))
fig.show()

n_per_h = len(df) // df["H"].nunique()
n_days = df["date"].nunique()
print(f"{n_per_h} измерений на высоту: {n_days} дней × 5 точек × 3 повтора = {n_per_h}")

print("\nОписательная статистика непрерывных ковариат:")
for col, unit in [("V", "м/с"), ("gusts", "м/с"), ("cloud", "%"), ("sat", "шт.")]:
    vals = df[col].dropna()
    print(f"  {col}: среднее = {vals.mean():.1f} {unit}, σ = {vals.std():.1f}, "
          f"диапазон [{vals.min():.1f} – {vals.max():.1f}]")

885 измерений на высоту: 59 дней × 5 точек × 3 повтора = 885

Описательная статистика непрерывных ковариат:
  V: среднее = 3.4 м/с, σ = 1.4, диапазон [0.7 – 8.7]
  gusts: среднее = 9.9 м/с, σ = 3.1, диапазон [3.1 – 19.2]
  cloud: среднее = 60.1 %, σ = 33.9, диапазон [0.0 – 100.0]
  sat: среднее = 22.4 шт., σ = 3.7, диапазон [16.0 – 31.0]


Таблица показывает монотонный рост среднего уровня ошибки: от $68{,}1$ мм на высоте 3 м до $303{,}1$ мм на высоте 20 м. Увеличение почти в 4,5 раза означает, что уже на уровне описательной статистики высота $H$ является основным кандидатом на главный фактор ошибки.

Одновременно увеличивается выборочное стандартное отклонение $s$: с $23{,}8$ мм на 3 м до $142{,}5$ мм на 20 м. Следовательно, на больших высотах система не только промахивается сильнее в среднем, но и даёт менее устойчивые повторные измерения.

Важно, что разброс растёт вместе с уровнем ошибки. Относительный разброс $s/\bar R$ меняется от $0{,}35$ на малых высотах до $0{,}47$ на высоте 20 м. В рассматриваемой задаче это означает, что крупная ошибка сопровождается крупной изменчивостью, поэтому далее нужно анализировать не только средние значения, но и форму распределений в высотных группах.

### Распределение ошибки по высотам

Среднее значение и стандартное отклонение показывают общий уровень ошибки, но не раскрывают форму распределения. Для проверки устойчивости вывода рассмотрим, как при увеличении $H$ меняются медиана, межквартильный размах и правый хвост распределения $R$.

In [160]:
fig = px.box(
    df, x="H", y="R", color="H_cat",
    title="Рис. 4. Распределение радиальной ошибки R по высоте полёта",
    labels={"H": "Высота H (м)", "R": "Радиальная ошибка R (мм)", "H_cat": "Высота"},
    category_orders={"H_cat": H_CAT_ORDER},
)
fig.update_layout(showlegend=False, height=500, width=850)
ALL_FIGS.append(("fig_04_boxplot_H", fig))
fig.show()

In [161]:
fig = px.violin(
    df, x="H", y="R", color="H_cat", box=True, points=False,
    title="Рис. 5. Форма распределения R по высотам (скрипичная диаграмма)",
    labels={"H": "Высота H (м)", "R": "Радиальная ошибка R (мм)", "H_cat": "Высота"},
    category_orders={"H_cat": H_CAT_ORDER},
)
fig.update_traces(width=0.85, spanmode="hard")
fig.update_layout(showlegend=False, height=600, width=900, violingap=0.15)
ALL_FIGS.append(("fig_05_violin_H", fig))
fig.show()

In [162]:
stats_by_h = df.groupby("H")["R"].agg(["mean", "std", "count"]).reset_index()
stats_by_h["se"] = stats_by_h["std"] / np.sqrt(stats_by_h["count"])
stats_by_h["ci95"] = stats_by_h["se"] * 1.96

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=stats_by_h["H"], y=stats_by_h["mean"],
    mode="lines+markers", name="Среднее R",
    line=dict(width=3), marker=dict(size=10),
))
fig.add_trace(go.Scatter(
    x=list(stats_by_h["H"]) + list(stats_by_h["H"][::-1]),
    y=list(stats_by_h["mean"] + stats_by_h["ci95"])
      + list((stats_by_h["mean"] - stats_by_h["ci95"])[::-1]),
    fill="toself", fillcolor="rgba(99,110,250,0.15)",
    line=dict(width=0), name="95% дов. интервал", showlegend=True,
))
fig.update_layout(
    title="Рис. 6. Средняя ошибка R по высоте (95% доверительный интервал, n=885 в группе)",
    xaxis_title="Высота H (м)", yaxis_title="R (мм)", height=400,
)
ALL_FIGS.append(("fig_06_mean_R_ci", fig))
fig.show()

Рис. 4 и 5 показывают не только рост центрального уровня ошибки, но и изменение всей формы распределения. Медиана $R$ смещается примерно от $65$ мм на 3 м до $264$ мм на 20 м, а межквартильный размах расширяется от 50–83 мм до 208–358 мм. Следовательно, на больших высотах увеличивается не только типичная ошибка, но и диапазон обычных, неэкстремальных значений.

Правый хвост распределения также становится длиннее: на высоте 20 м уже около $9{,}9\%$ наблюдений превышают 500 мм, тогда как на высотах до 10 м таких измерений нет. Эти точки не являются единственной причиной роста среднего: основной сдвиг виден уже по медиане и межквартильному размаху.

На рис. 6 средние значения возрастают монотонно, а 95% доверительные интервалы между высотами практически не перекрываются. Это означает, что различия между высотными группами нельзя объяснить только случайным разбросом повторных измерений.

Физически этот результат связан с геометрическим рычагом. При малой угловой нестабильности подвеса $\delta\varphi$ линейное смещение пятна можно приближённо записать как

$$
\Delta R \approx H \cdot \tan(\delta\varphi) \approx H \cdot \delta\varphi.
$$

Здесь $\Delta R$ — линейное смещение пятна на поверхности, $H$ — высота зависания, а $\delta\varphi$ — малая угловая ошибка подвеса. Формула показывает, что одна и та же угловая ошибка на большей высоте даёт больший промах. Поэтому рост $R$ с высотой является не только статистическим фактом, но и ожидаемым геометрическим следствием. Далее проверим, почему прирост особенно усиливается на участке 15–20 м.

### Влияние рельефа на высотах 15–20 м

Монотонный рост ошибки с высотой объясняется геометрическим рычагом, но участок 15–20 м требует отдельной проверки. Испытания проводились на территории карьера: до 10–15 м БПЛА находится внутри карьерной выемки, где рельеф частично экранирует ветровое воздействие, а при подъёме выше бровки попадает в более открытый поток.

Отсюда следует проверяемая гипотеза: если рельеф действительно ослабляет ветер на меньших высотах, то при переходе к 20 м ошибка должна расти быстрее, а различие между слабыми и сильными ветровыми режимами должно увеличиваться. Поэтому далее сравниваются медианные ошибки и приросты между соседними высотами.

In [163]:
v_q25 = df["V"].quantile(0.25)
v_q75 = df["V"].quantile(0.75)

low_wind = df[df["V"] <= v_q25].groupby("H")["R"].median().reset_index()
low_wind.columns = ["H", "R_median"]
low_wind["Режим"] = f"Слабый ветер (V ≤ {v_q25:.1f} м/с)"

high_wind = df[df["V"] >= v_q75].groupby("H")["R"].median().reset_index()
high_wind.columns = ["H", "R_median"]
high_wind["Режим"] = f"Сильный ветер (V ≥ {v_q75:.1f} м/с)"

plateau_df = pd.concat([low_wind, high_wind])

fig = px.line(
    plateau_df, x="H", y="R_median", color="Режим",
    markers=True,
    title="Рис. 7. Медианная ошибка R по высоте: слабый и сильный ветер",
    labels={"H": "Высота H (м)", "R_median": "Медианная R (мм)"},
)
fig.update_traces(line=dict(width=3), marker=dict(size=10))
fig.update_layout(height=450)
ALL_FIGS.append(("fig_07_plateau_wind", fig))
fig.show()

In [164]:
heights = sorted(df["H"].unique())
median_by_h = df.groupby("H")["R"].median()

deltas = []
for i in range(1, len(heights)):
    h_prev, h_curr = heights[i - 1], heights[i]
    delta = median_by_h[h_curr] - median_by_h[h_prev]
    deltas.append({"Интервал высот": f"{int(h_prev)}→{int(h_curr)} м", "ΔR": delta})

delta_df = pd.DataFrame(deltas)

fig = px.bar(
    delta_df, x="Интервал высот", y="ΔR", text="ΔR",
    title="Рис. 8. Прирост медианной ошибки между соседними высотами",
    labels={"Интервал высот": "Интервал высот (переход)", "ΔR": "ΔR (мм)"},
    color_discrete_sequence=["coral"],
)
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig.update_layout(height=400, yaxis_title="Прирост ΔR (мм)", xaxis_title="Интервал высот (переход)")
ALL_FIGS.append(("fig_08_delta_R", fig))
fig.show()

Рис. 7 и 8 согласуются с гипотезой о выходе БПЛА из ветровой тени карьера. Прирост медианной ошибки между соседними высотами составляет $15{,}8$ мм при переходе 3→5 м, $44{,}3$ мм при 5→10 м, $49{,}3$ мм при 10→15 м и уже $89{,}3$ мм при 15→20 м. Последний переход даёт наибольший прирост, то есть рост ошибки не замедляется, а ускоряется.

На рис. 7 линии слабого и сильного ветра расходятся с высотой. На малых высотах различие между режимами умеренное, а к 20 м разрыв становится максимальным: сильный ветер поднимает медианную ошибку примерно до 440 мм, тогда как при слабом ветре она остаётся около 200 мм.

Следовательно, на малых высотах ошибка в основном задаётся геометрическим рычагом $H$, а выше бровки карьера к нему добавляется ветровая составляющая. Для дальнейшего анализа это означает, что скорость ветра $V$ нужно рассматривать не только как самостоятельный фактор, но и во взаимодействии с высотой $H$.

### Влияние ветра

После высоты следующим фактором является скорость ветра $V$. Чем сильнее ветер, тем больше угловая нестабильность подвеса и тем вероятнее кратковременное смещение лазерного пятна. В рассматриваемой задаче это смещение усиливается высотой через связь $\Delta R \approx H\cdot\delta\varphi$. Поэтому $H$ и $V$ должны рассматриваться не порознь, а вместе: нас интересует как отдельная связь $R$ с $V$, так и будущий член взаимодействия $H \times V$ в регрессионной модели.

In [165]:
fig = px.scatter(
    df, x="V", y="R", color="H_cat",
    title="Рис. 9. Зависимость R от скорости ветра V (цвет — высота)",
    labels={"V": "Скорость ветра V (м/с)", "R": "R (мм)", "H_cat": "Высота"},
    category_orders={"H_cat": H_CAT_ORDER},
    opacity=0.35, trendline="ols",
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(height=500)
ALL_FIGS.append(("fig_09_R_vs_V", fig))
fig.show()

In [166]:
wind_bins = ["0.8–2.2", "2.2–2.9", "2.9–3.6", "3.6–4.4", "4.4–7.8"]

median_pivot = df.pivot_table(
    values="R", index="H", columns="V_bin", aggfunc="median", observed=False
).reindex(columns=wind_bins)
mean_pivot = df.pivot_table(
    values="R", index="H", columns="V_bin", aggfunc="mean", observed=False
).reindex(columns=wind_bins)

fig = make_subplots(
    rows=2,
    cols=1,
    vertical_spacing=0.18,
)

fig.add_trace(
    go.Heatmap(
        z=median_pivot.to_numpy(),
        x=median_pivot.columns.tolist(),
        y=median_pivot.index.tolist(),
        colorscale="YlOrRd",
        colorbar=dict(title="R (мм)", x=1.02, y=0.82, len=0.34),
        text=np.round(median_pivot.to_numpy(), 0),
        texttemplate="%{text:.0f}",
        hovertemplate="Высота H: %{y} м<br>Режим ветра: %{x}<br>Медиана R: %{z:.1f} мм<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Heatmap(
        z=mean_pivot.to_numpy(),
        x=mean_pivot.columns.tolist(),
        y=mean_pivot.index.tolist(),
        colorscale="Blues",
        colorbar=dict(title="R (мм)", x=1.02, y=0.18, len=0.34),
        text=np.round(mean_pivot.to_numpy(), 0),
        texttemplate="%{text:.0f}",
        hovertemplate="Высота H: %{y} м<br>Режим ветра: %{x}<br>Среднее R: %{z:.1f} мм<extra></extra>",
    ),
    row=2,
    col=1,
)

fig.update_layout(
    title="Рис. 10. Ошибка R (мм): медиана и среднее по высоте и режиму ветра",
    height=800,
)
fig.update_xaxes(side="top", row=1, col=1)
fig.update_xaxes(side="top", row=2, col=1)
fig.update_yaxes(
    title_text="Высота H (м)",
    autorange="reversed",
    tickmode="array",
    tickvals=[3, 5, 10, 15, 20],
    ticktext=["3", "5", "10", "15", "20"],
    row=1,
    col=1,
)
fig.update_yaxes(
    title_text="Высота H (м)",
    autorange="reversed",
    tickmode="array",
    tickvals=[3, 5, 10, 15, 20],
    ticktext=["3", "5", "10", "15", "20"],
    row=2,
    col=1,
)
ALL_FIGS.append(("fig_10_heatmap_HV", fig))
fig.show()

На рис. 9 все линии тренда имеют положительный наклон: при увеличении скорости ветра $V$ радиальная ошибка $R$ возрастает на каждой высоте. Однако наклон для 20 м заметно больше, чем для 3–5 м: оценка прироста ошибки на 1 м/с ветра составляет около $6$ мм/(м/с) на высоте 3 м и около $77$ мм/(м/с) на высоте 20 м. Иными словами, один и тот же прирост ветра даёт более чем десятикратно более крупный промах на большой высоте.

Рис. 10 показывает тот же вывод в агрегированном виде. В зоне 3 м и слабого ветра медианная ошибка составляет около 45 мм, а в зоне 20 м и сильного ветра — около 463 мм. Такой рост возникает не из-за одного фактора, а из-за их совместного действия.

По вертикали тепловой карты изменение сильнее, чем по горизонтали: переход к большей высоте увеличивает ошибку заметнее, чем переход к соседнему ветровому режиму. Следовательно, графики подтверждают рабочую иерархию $H > V$, но одновременно требуют проверить регрессионный член $H \times V$, отвечающий за совместное действие высоты и ветра.

### Вторичные признаки: направление ветра, спутники и облачность

Высота $H$ и скорость ветра $V$ задают основную физическую картину, но в журнале также фиксировались направление ветра, число видимых спутников GNSS $N_{\text{sat}}$ и облачность $C$. Эти признаки проверяются как контрольные: они помогают убедиться, что итоговая модель по $H$ и $V$ не подменяет физический механизм случайным сопутствующим показателем.

In [167]:
sample_rxy = df.sample(n=min(2000, len(df)), random_state=42)

fig = px.scatter(
    sample_rxy, x="r_x", y="r_y", color="wind_dir",
    color_continuous_scale="HSV",
    title="Рис. 11. Компоненты ошибки r_x и r_y (цвет — направление ветра)",
    labels={"r_x": "r_x (мм)", "r_y": "r_y (мм)", "wind_dir": "Направл. (°)"},
    opacity=0.5,
)
fig.update_traces(marker=dict(size=4))

r_max = max(sample_rxy["r_x"].abs().max(), sample_rxy["r_y"].abs().max()) * 1.05
radii = np.linspace(r_max * 0.25, r_max, 4)
theta_circle = np.linspace(0, 2 * np.pi, 361)

for r in radii:
    fig.add_trace(go.Scatter(
        x=r * np.cos(theta_circle), y=r * np.sin(theta_circle),
        mode="lines", line=dict(color="gray", width=0.5, dash="dot"),
        showlegend=False, hoverinfo="skip",
    ))

compass = {"С": 90, "СВ": 45, "В": 0, "ЮВ": 315, "Ю": 270, "ЮЗ": 225, "З": 180, "СЗ": 135}
for label, deg in compass.items():
    rad = np.radians(deg)
    fig.add_trace(go.Scatter(
        x=[0, r_max * np.cos(rad)], y=[0, r_max * np.sin(rad)],
        mode="lines", line=dict(color="lightgray", width=0.5),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_annotation(
        x=r_max * 1.08 * np.cos(rad), y=r_max * 1.08 * np.sin(rad),
        text=label, showarrow=False, font=dict(size=10, color="gray"),
    )

fig.update_layout(
    height=620, width=650,
    xaxis=dict(scaleanchor="y", range=[-r_max * 1.15, r_max * 1.15]),
    yaxis=dict(range=[-r_max * 1.15, r_max * 1.15]),
)
ALL_FIGS.append(("fig_11_rxy_winddir", fig))
fig.show()

На рис. 11 облако точек $(r_x, r_y)$ в центральной области близко к радиально симметричному: ошибки распределены вокруг начала координат без устойчивого вытягивания в одном направлении. Цвет, соответствующий азимуту ветра, перемешан по квадрантам, поэтому выраженной направленной связи между вектором ошибки и направлением ветра не наблюдается.

Для дальнейшей модели это означает, что направление ветра не требуется вводить как отдельную векторную пару признаков. В рамках имеющихся данных достаточно использовать скалярную скорость $V$, которая отражает силу ветрового воздействия на подвес и уже связана с ростом $R$.

Помимо вектора ветра, в журнале фиксировался навигационный параметр $N_{\text{sat}}$ — число видимых спутников GNSS. Связь этого показателя с ошибкой проверяется и на корреляционной диаграмме (рис. 12), и в дисперсионном анализе как контрольная ковариата. В наблюдавшемся диапазоне $N_{\text{sat}}$ лежит в пределах 16–31 при медиане 23, то есть устойчиво превышает рабочий минимум RTK-режима. Поэтому число спутников отражает скорее фон навигационных условий, чем управляемый фактор полёта. Перед формальной проверкой сначала рассмотрим общую корреляционную картину.

In [168]:
factors = ["H", "V", "sat", "cloud"]
factor_names = {"H": "Высота", "V": "Ветер", "sat": "Спутники", "cloud": "Обл."}
corrs = []
for f in factors:
    rho, p = stats.spearmanr(df[f], df["R"])
    corrs.append({
        "Фактор": factor_names[f],
        "ρ Спирмена": rho,
        "p-значение": p,
        "|ρ|": abs(rho)
    })

corr_df = pd.DataFrame(corrs)

fig = px.bar(
    corr_df, x="Фактор", y="ρ Спирмена", text="ρ Спирмена",
    color="|ρ|", color_continuous_scale="Blues",
    title="Рис. 12. Ранговая корреляция Спирмена факторов с R",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(height=400, coloraxis_showscale=False, yaxis_title="ρ Спирмена")
ALL_FIGS.append(("fig_12_spearman", fig))
fig.show()

Рис. 12 сводит предварительный графический анализ в одну корреляционную картину. Для ранговой корреляции Спирмена исходные значения заменяются их рангами, после чего рассчитывается корреляция между ранговыми рядами:

$$
\rho_s = 1 - \frac{6\sum_{i=1}^{n} d_i^2}{n(n^2-1)}.
$$

Здесь $d_i$ — разность рангов двух признаков для $i$-го наблюдения. Иными словами, $\rho_s$ показывает, насколько согласованно признаки возрастают или убывают, не требуя линейной зависимости на исходной шкале. Значения $\rho_s$ лежат в интервале от $-1$ до $1$: $\rho_s = 1$ соответствует строгому монотонному возрастанию, $\rho_s = -1$ — строгому убыванию, $\rho_s = 0$ — отсутствию монотонной связи.

Самая сильная монотонная связь с ошибкой наблюдается у высоты: $\rho_H = 0{,}825$. Ветер занимает второе место: $\rho_V = 0{,}401$. Число спутников GNSS даёт слабую отрицательную связь $\rho_{N_{\text{sat}}} = -0{,}246$: при большем числе видимых спутников ошибка в среднем чуть ниже. Облачность даёт положительную корреляцию $\rho_C = 0{,}249$, но этот результат нельзя читать как самостоятельное физическое влияние облаков на лазерное наведение: в рассматриваемой выборке пасмурные дни чаще сопровождаются более сильным ветром.

Итак, графический и корреляционный анализ задают предварительную иерархию факторов: высота — основной источник изменения $R$, ветер — второй, число спутников и облачность требуют отдельной проверки как сопутствующие признаки. Прежде чем перейти к формальному дисперсионному анализу, отдельно посмотрим на связь $R$ и $N_{\text{sat}}$.

In [169]:
sat_profile = df.groupby("sat", as_index=False).agg(
    median_R=("R", "median"),
    count=("R", "size"),
)
sample_sat = df.sample(n=min(2500, len(df)), random_state=42)

fig = px.scatter(
    sample_sat,
    x="sat",
    y="R",
    color="H_cat",
    category_orders={"H_cat": H_CAT_ORDER},
    opacity=0.35,
    title="Рис. 13. Радиальная ошибка R при разном числе спутников GNSS",
    labels={"sat": "Спутники GNSS", "R": "R (мм)", "H_cat": "Высота"},
)
fig.update_traces(marker=dict(size=4))
fig.add_trace(go.Scatter(
    x=sat_profile["sat"],
    y=sat_profile["median_R"],
    mode="lines+markers",
    line=dict(color="black", width=3),
    marker=dict(size=7),
    name="Медиана R по sat",
    hovertemplate="sat=%{x}<br>медиана R=%{y:.1f} мм<extra></extra>",
))
fig.update_layout(height=520, width=760)
ALL_FIGS.append(("fig_13_sat_vs_R", fig))
fig.show()

Рис. 13 является прямой визуальной проверкой связи между $N_{\text{sat}}$ и ошибкой. На рис. 12 эта связь видна как отрицательная ранговая корреляция: при большем числе спутников $R$ в среднем снижается. Однако сама по себе такая корреляция ещё не означает, что число спутников должно войти в итоговое уравнение прогноза.

На рис. 13 медианная линия действительно постепенно убывает: при малом $N_{\text{sat}}$ типичная ошибка выше, чем при большем числе спутников. Но внутри каждого уровня остаётся широкий вертикальный разброс, а цветовая разбивка показывает, что значительная часть этого разброса связана с высотой. Иными словами, $N_{\text{sat}}$ фиксирует навигационный фон условий измерения, но не заменяет основные управляемые факторы — высоту $H$ и скорость ветра $V$.

Следовательно, число спутников сохраняется в дисперсионном анализе как контрольная ковариата, но в прогнозную регрессионную модель не включается. Такой переход позволяет учесть навигационный фон и одновременно не перегружать итоговое эксплуатационное уравнение.


### Проверка предпосылок дисперсионного анализа

Графики уже показали, что ошибка растёт с высотой и ветром, но перед дисперсионным анализом необходимо проверить, насколько данные согласуются с его базовыми предпосылками. Здесь важны две проверки: форма распределения внутри высотных групп и однородность дисперсий между группами.

Критерий Шапиро-Уилка проверяет нулевую гипотезу

$$
H_0: R \text{ в группе имеет нормальное распределение}.
$$

Для количественной проверки используется статистика

$$
W = \frac{\left(\sum_{i=1}^{n} a_i x_{(i)}\right)^2}{\sum_{i=1}^{n}(x_i - \bar{x})^2}.
$$

Здесь $x_{(i)}$ — упорядоченные значения выборки, $\bar{x}$ — среднее значение, а коэффициенты $a_i$ задаются ожидаемыми порядковыми статистиками нормального распределения. Числитель показывает, насколько порядок наблюдений похож на нормальный, а знаменатель нормирует это сравнение общим разбросом выборки. Статистика $W$ лежит в интервале от 0 до 1: чем ближе $W$ к единице, тем ближе выборка к нормальному закону. Значение $p$ здесь не следует читать изолированно: при большом $n$ даже умеренные отклонения становятся формально значимыми.

In [170]:
H_ORDER = sorted(df["H"].unique())
groups_by_H = [df.loc[df["H"] == h, "R"].values for h in H_ORDER]

shapiro_rows = []
for h, grp in zip(H_ORDER, groups_by_H):
    sample = (
        grp if len(grp) <= 5000
        else np.random.default_rng(42).choice(grp, 5000, replace=False)
    )
    W, p = stats.shapiro(sample)
    shapiro_rows.append([
        int(h), len(grp), round(W, 4), f"{p:.2e}",
        "Норм." if p > ALPHA else f"W={W:.4f}"
    ])

fig = go.Figure(data=[go.Table(
    columnwidth=[60, 60, 80, 100, 120],
    header=dict(
        values=["<b>H (м)</b>", "<b>N</b>", "<b>W</b>", "<b>p-значение</b>", "<b>Заключение</b>"],
        fill_color="#34495e", font=dict(color="white", size=13),
        align="center", height=32,
    ),
    cells=dict(
        values=list(zip(*shapiro_rows)),
        fill_color=[["#ecf0f1", "#ffffff"] * 3],
        font=dict(size=12), align="center", height=28,
    )
)])
fig.update_layout(
    title="Таблица 2. Результаты критерия Шапиро–Уилка по высотным группам",
    width=700, height=260, margin=dict(l=10, r=10, t=50, b=10),
    font=dict(family="Arial"),
)
ALL_FIGS.append(("fig_shapiro_table", fig))
fig.show()

В каждой высотной группе содержится $n = 885$ измерений, поэтому критерий Шапиро-Уилка обладает высокой мощностью. Значения $W$ снижаются от $0{,}976$ на 3 м до $0{,}872$ на 20 м. Это означает, что распределения сохраняют нормальное ядро, но на больших высотах усиливаются правая асимметрия и влияние хвостов.

Формально $p$-значения малы во всех группах, однако это не делает дальнейший анализ непригодным. При таком объёме выборки важнее учитывать характер отклонения: оно ожидаемо для величины $R \geq 0$ и связано с редкими крупными промахами. Поэтому результат критерия используется как диагностический сигнал, а не как единственное основание для отказа от дисперсионного анализа.

---

#### Однородность дисперсий. Критерий Левена

Следующая предпосылка — равенство дисперсий между высотными группами. Критерий Левена проверяет гипотезу

$$
H_0\colon \sigma^2_1 = \sigma^2_2 = \ldots = \sigma^2_k,
$$

то есть равенство дисперсий во всех сравниваемых группах. В варианте Brown-Forsythe сначала рассчитываются отклонения наблюдений от медианы своей группы:

$$
Z_{ij}=|R_{ij}-\tilde R_j|.
$$

Здесь $R_{ij}$ — $i$-е наблюдение в $j$-й высотной группе, а $\tilde R_j$ — медиана этой группы. Затем для величин $Z_{ij}$ выполняется однофакторный дисперсионный анализ. Иными словами, использование медианы вместо среднего делает критерий устойчивым к асимметрии распределения, а $R$ как раз имеет правый хвост. В этой задаче проверка особенно важна, потому что ранее уже было видно: разброс ошибки растёт вместе с её средним уровнем.

In [171]:
lev_stat, lev_p = stats.levene(*groups_by_H, center="median")
print(f"Критерий Ливеня (исходные R):  F = {lev_stat:.2f},  p = {lev_p:.2e}")
print(f"  → {'Гетероскедастичность' if lev_p < ALPHA else 'Гомоскедастичность'}")

Критерий Ливеня (исходные R):  F = 270.12,  p = 5.49e-208
  → Гетероскедастичность


На исходной шкале $R$ критерий Левена показывает статистически значимое различие дисперсий высотных групп. Этот результат не выглядит артефактом: в описательной статистике уже было видно, что абсолютный разброс растёт вместе с уровнем ошибки.

Следовательно, неоднородность дисперсий следует понимать как физическое свойство процесса, а не как ошибку измерений. Чем выше ожидаемая ошибка, тем шире диапазон возможных промахов, особенно при порывистом ветре. Поэтому дальнейшие выводы должны опираться не только на формальные $p$-значения, но и на размер эффекта, физическую интерпретацию и диагностику остатков.

---

#### Квантиль-квантильный график (QQ-график) остатков

Дополнительно проверим форму остатков на QQ-графике. Для каждого наблюдения остаток определяется как

$$
e_i = R_i - \hat R_i.
$$

Здесь $R_i$ — измеренная радиальная ошибка, а $\hat R_i$ — значение, рассчитанное основной линейной моделью $R \sim H + V + H{:}V$. QQ-график сопоставляет наблюдаемые квантили остатков с квантилями нормального распределения. Если точки идут вдоль теоретической прямой, нормальная аппроксимация описывает основную массу остатков; систематические отклонения на краях указывают на тяжёлые хвосты.

In [172]:
reg_model_qq = ols("R ~ H + V + H:V", data=df).fit()
residuals = reg_model_qq.resid

qq_data = stats.probplot(residuals, dist="norm")
theoretical = qq_data[0][0]
observed = qq_data[0][1]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=theoretical, y=observed, mode="markers",
    marker=dict(size=3, color="steelblue", opacity=0.4), name="Остатки",
))
rng_line = [min(theoretical), max(theoretical)]
fig.add_trace(go.Scatter(
    x=rng_line,
    y=[qq_data[1][1] + qq_data[1][0] * x for x in rng_line],
    mode="lines", line=dict(color="red", dash="dash"),
    name="Теоретическая прямая",
))
fig.update_layout(
    title="Рис. 14. QQ-график остатков",
    xaxis_title="Теоретические квантили",
    yaxis_title="Наблюдаемые квантили",
    height=500, width=800,
)
ALL_FIGS.append(("fig_14_qq_plot", fig))
fig.show()

На рис. 14 центральная часть остатков хорошо следует теоретической прямой: для основной массы наблюдений нормальная аппроксимация приемлема. Отклонения заметны на краях, особенно в правом хвосте, где наблюдаемые квантили уходят выше теоретических.

Это означает, что в данных есть редкие крупные ошибки, более выраженные, чем ожидалось бы при идеально нормальных остатках. С практической точки зрения такие точки согласуются с физикой эксперимента: сильные порывы ветра и кратковременные колебания подвеса создают большие промахи.

Следовательно, предпосылки ANOVA выполняются не идеально, но характер отклонений понятен и ожидаем для полевых данных. При объёме $N = 4425$ дисперсионный анализ остаётся пригодным для оценки силы факторов, если интерпретировать не только $p$-значения, но и размеры эффектов, а также проверять регрессионные остатки.

### Дисперсионный анализ

Дисперсионный анализ нужен для того, чтобы разделить общий разброс радиальной ошибки $R$ на объяснённые части и остаток. Иными словами, он отвечает не только на вопрос «значим ли фактор», но и на вопрос «какую долю изменчивости ошибки он действительно объясняет». Такая логика соответствует схеме: сначала выделяются факторы, затем оценивается их вклад, после чего формируется регрессионное уравнение для практического расчёта.

#### Ковариационный анализ (тип II)

Для совместной проверки основных и вторичных признаков используется модель

$$
R_{ij} = \mu + \alpha_{H_i} + \beta_1 V + \beta_2 N_{\text{sat}} + \beta_3 C + \varepsilon_{ij}.
$$

Здесь $\mu$ задаёт общий уровень ошибки, относительно которого оцениваются остальные влияния. Член $\alpha_{H_i}$ — поправка для $i$-й высотной группы, $i = 1\ldots 5$; высота вводится как категориальный фактор, чтобы не навязывать ей линейную форму. Коэффициент $\beta_1$ описывает вклад скорости ветра, $\beta_2$ — связь с числом видимых спутников GNSS, $\beta_3$ — связь с облачностью. Остаток $\varepsilon_{ij}$ содержит ту часть ошибки, которую эти факторы не объясняют.

$N_{\text{sat}}$ и $C$ вводятся именно как контрольные ковариаты: их роль — проверить, что эффект $H$ и $V$ не объясняется навигационными или сопутствующими погодными условиями. Это не означает, что они автоматически попадут в прогнозное уравнение. Для каждого фактора рассчитываются две меры силы эффекта:

$$
\eta^2_{\text{partial}} = \frac{SS_{\text{фактор}}}{SS_{\text{фактор}} + SS_{\text{остаток}}},
$$

$$
\omega^2 = \frac{SS_{\text{фактор}} - df_{\text{фактор}} \cdot MS_{\text{остаток}}}{SS_{\text{фактор}} + (N - df_{\text{фактор}}) \cdot MS_{\text{остаток}}}.
$$

В этих формулах $SS$ — сумма квадратов, то есть часть общего разброса, связанная с фактором или остатком; $df$ — число степеней свободы; $MS$ — средний квадрат остатка. Частичная $\eta^2$ показывает долю разброса, объяснённую фактором относительно фактора и остатка, а $\omega^2$ оценивает вклад более консервативно, с поправкой на число степеней свободы. Поэтому именно $\omega^2$ далее принимается как основная мера силы фактора.

In [173]:
df["H_factor"] = df["H"].astype(str)

model_full = ols("R ~ C(H_factor) + V + sat + cloud", data=df).fit()
aov = anova_lm(model_full, typ=2)

ss_resid = aov.loc["Residual", "sum_sq"]
df_resid = aov.loc["Residual", "df"]
ms_resid = ss_resid / df_resid
n = len(df)

results = []
for factor in ["C(H_factor)", "V", "sat", "cloud"]:
    if factor not in aov.index:
        continue
    ss = aov.loc[factor, "sum_sq"]
    df_f = aov.loc[factor, "df"]
    f_val = aov.loc[factor, "F"]
    p_val = aov.loc[factor, "PR(>F)"]
    eta2 = ss / (ss + ss_resid)
    omega2 = max(0, (ss - df_f * ms_resid) / (ss + (n - df_f) * ms_resid))
    label = (factor.replace("C(", "").replace("_factor)", "").replace(")", "")
              .replace("sat", "Спутники").replace("cloud", "Облачность"))
    results.append({
        "Фактор": label, "SS": f"{ss:.0f}", "df": int(df_f),
        "F": f"{f_val:.1f}", "p-значение": f"{p_val:.2e}",
        "η²": f"{eta2:.4f}", "ω²": f"{omega2:.4f}",
        "_omega2": omega2, "_eta2": eta2, "_label": label,
    })

anova_df = pd.DataFrame(results)
print("Таблица 3. Ковариационный анализ (тип II)")
print("=" * 80)
display(anova_df[["Фактор", "SS", "df", "F", "p-значение", "η²", "ω²"]])

print(f"\nR² модели: {model_full.rsquared:.4f}")
print(f"R² скорректированный: {model_full.rsquared_adj:.4f}")

Таблица 3. Ковариационный анализ (тип II)


,Фактор,SS,df,F,p-значение,η²,ω²
0,H,31300798,4,2353.9,0.00e+00,0.6807,0.6802
1,V,8253799,1,2482.9,0.00e+00,0.3598,0.3593
2,Спутники,780231,1,234.7,1.17e-51,0.0505,0.0502
3,Облачность,433797,1,130.5,8.35e-30,0.0287,0.0284



R² модели: 0.7598
R² скорректированный: 0.7595


In [174]:
fig = px.bar(
    anova_df, x="_label", y="_omega2", text="ω²",
    title="Рис. 15. Сила влияния факторов (ω²)",
    labels={"_label": "Фактор", "_omega2": "ω²"},
    color="_omega2", color_continuous_scale="Teal",
)
fig.update_traces(textposition="outside")
fig.update_layout(height=400, coloraxis_showscale=False, yaxis_title="ω²")
ALL_FIGS.append(("fig_15_omega2", fig))
fig.show()

In [175]:
# В таблице ANOVA η² — частичная: SS/(SS+SS_resid); такие величины не суммируются в 1,
# поэтому 1 - sum(η²) часто ≤ 0 и срез «остаток» на круге не появляется.
# Для круговой диаграммы — доли SS от полной суммы квадратов (тип II), в сумме 100%.
ss_total = float(aov["sum_sq"].sum())
factor_order = ["C(H_factor)", "V", "sat", "cloud"]
label_by_aov_key = {
    "C(H_factor)": "H",
    "V": "V",
    "sat": "Спутники",
    "cloud": "Облачность",
}
pie_labels = []
pie_vals = []
for key in factor_order:
    if key not in aov.index:
        continue
    pie_labels.append(label_by_aov_key[key])
    pie_vals.append(float(aov.loc[key, "sum_sq"]) / ss_total)
pie_labels.append("Необъяснённая часть")
pie_vals.append(float(aov.loc["Residual", "sum_sq"]) / ss_total)

fig = go.Figure(data=[go.Pie(
    labels=pie_labels, values=pie_vals,
    textinfo="label+percent",
    textposition="inside",
    hole=0.3,
    sort=False,
    marker=dict(colors=["#2ecc71", "#3498db", "#9b59b6", "#e74c3c", "#bdc3c7"]),
)])
fig.update_layout(
    title="Рис. 16. Доли SS в ANOVA: факторы и остаток",
    height=480, width=760,
    font=dict(size=13),
)
ALL_FIGS.append(("fig_16_pie_ss", fig))
fig.show()

Таблица ANOVA показывает, что формальная значимость факторов сама по себе недостаточна для выбора регрессионной модели. При большом объёме выборки малые $p$-значения могут появляться и у слабых сопутствующих признаков, поэтому главный интерес представляет размер эффекта $\omega^2$.

Высота $H$ имеет наибольшую силу эффекта: $\omega^2_H \approx 0{,}68$. Это согласуется с физикой геометрического рычага: при увеличении $H$ одна и та же угловая ошибка подвеса превращается в больший линейный промах.

Скорость ветра $V$ занимает второе место: $\omega^2_V \approx 0{,}36$. Ветер раскачивает платформу и подвес, поэтому его вклад особенно важен в сочетании с высотой, что уже было видно на тепловой карте $H \times V$.

Число спутников и облачность дают заметно меньший вклад: $\omega^2_{N_{\text{sat}}} \approx 0{,}050$, $\omega^2_C \approx 0{,}028$. Для спутников причина методическая: число видимых аппаратов — это лишь один из частных индикаторов условий приёма, тогда как точность позиционирования определяется геометрией созвездия, статусом RTK-решения, отношением сигнал/шум, многолучёвостью и состоянием атмосферы. Поэтому одно лишь $N_{\text{sat}}$, оторванное от $\text{DOP}$ и статуса Fix/Float, отражает только грубые провалы условий приёма. Облачность рассматривается как сопутствующий погодный признак: её возможная связь с $R$ объясняется тем, что пасмурные дни чаще сопровождаются более сильным ветром.

Рис. 15 фиксирует иерархию факторов по $\omega^2$, а рис. 16 показывает доли сумм квадратов в дисперсионном разложении, включая вторичные признаки. Эти величины не следует отождествлять с $R^2$ регрессионной модели ниже, потому что спецификации моделей различаются.

Следовательно, в регрессионное моделирование включаются только высота $H$ и скорость ветра $V$. Дисперсионный анализ выполнил роль отбора факторов; теперь построим уравнение, по которому можно прогнозировать $R$ и задавать практическую область допустимых режимов.

---
### Регрессионная модель с взаимодействием $H \times V$

Дисперсионный анализ показал, какие факторы имеют самостоятельный вклад, но не даёт удобного уравнения для расчёта ожидаемой ошибки. Поэтому следующим шагом строится регрессионная модель, связывающая радиальную ошибку $R$ с высотой $H$ и скоростью ветра $V$.

В основную модель включён член взаимодействия $H \times V$. Он нужен не как формальное усложнение, а как отражение физического механизма: на большей высоте та же ветровая угловая нестабильность подвеса превращается в больший линейный промах.

Модель имеет вид

$$
R = \beta_0 + \beta_H H + \beta_V V + \beta_{HV}HV + \varepsilon.
$$

Здесь $\beta_0$ задаёт базовый уровень ошибки, относительно которого оцениваются остальные влияния. Член $\beta_H H$ описывает вклад высоты при фиксированной скорости ветра, $\beta_V V$ — вклад скорости ветра при фиксированной высоте. Произведение $HV$ выделено отдельно, потому что действие ветра зависит от высоты. Остаток $\varepsilon$ включает кратковременные порывы, вибрации подвеса и прочие непредсказанные отклонения.

Параметры оцениваются методом наименьших квадратов:

$$
\hat\beta=(X^TX)^{-1}X^Ty.
$$

Здесь $X$ — матрица наблюдений по предикторам, $y$ — вектор измеренных значений $R$, а $\hat\beta$ — вектор оценённых коэффициентов. Оценка выбирается так, чтобы минимизировать суммарный квадрат отклонений между наблюдаемыми и предсказанными $R$. В прикладном смысле модель описывает условное среднее $E(R\mid H,V)$, то есть ожидаемую радиальную ошибку при заданной высоте и скорости ветра.

Качество модели оценивается коэффициентом детерминации $R^2$, то есть долей изменчивости $R$, объяснённой уравнением. Скорректированный коэффициент $R^2_{\text{adj}}$ используется вместе с ним, потому что учитывает число предикторов и поэтому удобен для сравнения моделей разной сложности.

In [176]:
reg_model = ols("R ~ H + V + H:V", data=df).fit()

coef_df = pd.DataFrame({
    "Коэффициент": reg_model.params,
    "Ст. ошибка": reg_model.bse,
    "t-стат.": reg_model.tvalues,
    "p-значение": reg_model.pvalues,
    "Нижн. 95%": reg_model.conf_int()[0],
    "Верхн. 95%": reg_model.conf_int()[1],
}).round(4)

print(f"R² = {reg_model.rsquared:.4f},  R² скорр. = {reg_model.rsquared_adj:.4f}")
print(f"F = {reg_model.fvalue:.1f},  p(F) = {reg_model.f_pvalue:.2e}")
print()
display(coef_df)

R² = 0.8139,  R² скорр. = 0.8138
F = 6446.0,  p(F) = 0.00e+00



,Коэффициент,Ст. ошибка,t-стат.,p-значение,Нижн. 95%,Верхн. 95%
Intercept,40.7422,3.9242,10.3823,0.0000,33.0488,48.4356
H,-1.0656,0.3201,-3.3293,0.0009,-1.6932,-0.4381
V,-6.6104,1.0754,-6.1468,0.0000,-8.7188,-4.5020
H:V,4.1723,0.0869,48.0254,0.0000,4.0020,4.3427


Полученная линейная модель на исходной шкале ошибки записывается как

$$
\hat{R} = 40{,}74 - 1{,}07H - 6{,}61V + 4{,}17HV.
$$

Здесь $\hat R$ выражается в миллиметрах, $H$ — в метрах, а $V$ — в м/с. Отдельные коэффициенты при $H$ и $V$ в модели с взаимодействием нельзя рассматривать изолированно: их смысл задаётся вместе с членом $HV$. Положительный коэффициент при $HV$ означает, что влияние ветра усиливается с высотой. Иными словами, чем выше БПЛА, тем больший линейный промах даёт та же ветровая угловая нестабильность подвеса.

In [177]:
show_eq(reg_model)

$$\hat{R} = 40.7 -1.07\,H -6.61\,V +4.17\,H\!\cdot\!V \quad (R^2 = 0.814)$$

In [178]:
plot_coefs = coef_df.drop("Intercept", errors="ignore").reset_index()
plot_coefs.columns = ["Фактор"] + list(plot_coefs.columns[1:])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=plot_coefs["Коэффициент"], y=plot_coefs["Фактор"],
    mode="markers", marker=dict(size=10, color="steelblue"),
    error_x=dict(
        type="data", symmetric=False,
        array=(plot_coefs["Верхн. 95%"] - plot_coefs["Коэффициент"]).values,
        arrayminus=(plot_coefs["Коэффициент"] - plot_coefs["Нижн. 95%"]).values,
    ),
    name="Коэффициент",
))
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Рис. 17. Коэффициенты регрессии (с 95% доверительным интервалом)",
    xaxis_title="Значение коэффициента", height=350,
)
ALL_FIGS.append(("fig_17_coef_forest", fig))
fig.show()

Рис. 17 показывает знаки и доверительные интервалы коэффициентов основной линейной модели. Положительный коэффициент при $H:V$ не пересекает нуль, поэтому взаимодействие высоты и ветра статистически подтверждается: ветер действительно становится опаснее на больших высотах.

Абсолютные значения коэффициентов напрямую сравнивать нельзя, потому что $H$, $V$ и $H\times V$ имеют разные размерности. Поэтому вопрос «какой фактор сильнее» решается не по длине точки на рис. 17, а по $\omega^2$ из дисперсионного анализа. В этом смысле рис. 17 отвечает за направление и форму уравнения, а рис. 15 — за ранжирование факторов по практическому вкладу.

In [179]:
df["R_pred"] = reg_model.predict(df)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["R_pred"], y=df["R"], mode="markers",
    marker=dict(size=3, opacity=0.3, color="steelblue"),
    name="Измерения",
))
r_range = [0, max(df["R"].max(), df["R_pred"].max()) * 1.05]
fig.add_trace(go.Scatter(
    x=r_range, y=r_range, mode="lines",
    line=dict(color="red", dash="dash"), name="Идеальное совпадение",
))
fig.update_layout(
    title="Рис. 18. Предсказанные значения R и наблюдаемые",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="Наблюдаемое R (мм)",
    height=500, width=700,
)
ALL_FIGS.append(("fig_18_pred_vs_obs", fig))
fig.show()

На рис. 18 красная пунктирная линия соответствует идеальному совпадению предсказанного и наблюдаемого $R$. Основное облако точек вытянуто вдоль этой линии: при малых предсказанных ошибках наблюдаемые значения также малы, а при больших — возрастают.

Однако в правой части графика разброс заметно увеличивается. При $\hat R$ порядка 300–500 мм наблюдаемые значения могут уходить значительно выше линии идеального совпадения. Иными словами, модель хорошо описывает общий тренд, но хуже воспроизводит редкие пиковые ошибки, возникающие при неблагоприятном сочетании высоты и ветра.

Для исследования это ограничение важно зафиксировать отдельно: уравнение пригодно для оценки ожидаемой ошибки, но не должно интерпретироваться как точный прогноз каждого единичного промаха. Следующий диагностический график проверяет это ограничение через остатки модели.

In [180]:
fitted = reg_model.fittedvalues
resid = reg_model.resid

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fitted, y=resid, mode="markers",
    marker=dict(size=3, opacity=0.3, color="steelblue"),
    name="Остатки",
))
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.update_layout(
    title="Рис. 19. Остатки модели и предсказанные значения",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="Остаток (мм)",
    height=500, width=750,
)
ALL_FIGS.append(("fig_19_resid_vs_fitted", fig))
fig.show()

Рис. 19 показывает остатки $e_i = R_i - \hat R_i$ в зависимости от предсказанного значения. Если бы дисперсия ошибки была постоянной, точки образовывали бы примерно одинаковую вертикальную полосу вокруг нуля на всём диапазоне $\hat R$.

Вместо этого виден веерообразный рисунок. При малых предсказанных ошибках остатки компактны, а при $\hat R$ выше 300 мм разброс резко увеличивается и появляются крупные положительные остатки. Систематического смещения всего облака относительно нуля нет: модель в среднем не завышает и не занижает ошибку. Проблема состоит именно в росте разброса.

Такой график подтверждает вывод критерия Левена: ошибка имеет гетероскедастичную, пропорциональную природу. Чем выше ожидаемая ошибка, тем шире диапазон возможных отклонений от прогноза. Следующий график показывает тот же эффект в стандартной диагностической форме scale-location.

In [181]:
std_resid = resid / resid.std()
sqrt_abs_std = np.sqrt(np.abs(std_resid))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fitted, y=sqrt_abs_std, mode="markers",
    marker=dict(size=3, opacity=0.3, color="steelblue"),
    showlegend=False,
))

n_bins = 30
bin_edges = np.linspace(fitted.min(), fitted.max(), n_bins + 1)
bin_centers, bin_means = [], []
for j in range(n_bins):
    mask = (fitted >= bin_edges[j]) & (fitted < bin_edges[j + 1])
    if mask.sum() > 10:
        bin_centers.append((bin_edges[j] + bin_edges[j + 1]) / 2)
        bin_means.append(sqrt_abs_std[mask].mean())

fig.add_trace(go.Scatter(
    x=bin_centers, y=bin_means, mode="lines",
    line=dict(color="red", width=2.5), name="Тренд",
))
fig.update_layout(
    title="Рис. 20. Однородность разброса остатков",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="√|стандартизованный остаток|",
    height=480, width=750,
)
ALL_FIGS.append(("fig_20_scale_location", fig))
fig.show()

На рис. 20 по вертикали отложена величина $\sqrt{|e_i^*|}$, где $e_i^* = e_i / \hat\sigma$ — стандартизованный остаток. Если разброс остатков постоянен, красная линия тренда должна быть примерно горизонтальной.

Здесь линия растёт вместе с предсказанным $R$. Это означает, что абсолютная ошибка модели увеличивается при увеличении уровня отклика: чем больше ожидаемая радиальная ошибка, тем шире разброс наблюдений вокруг неё.

Для дальнейшей интерпретации это важно как ограничение линейной модели на исходной шкале миллиметров: она хорошо описывает средний тренд, но хуже воспроизводит редкие крупные промахи. Поэтому сравнение ниже выполняется между двумя линейными спецификациями на одной и той же шкале: $H+V$ и $H+V+H\times V$.

#### Интерпретация модели с взаимодействием

Линейная модель с $H \times V$ описывает радиальную ошибку на исходной шкале миллиметров. Положительный член $HV$ показывает усиление ветрового влияния с высотой: один и тот же прирост скорости ветра даёт разный прирост ошибки на 3 м и на 20 м.

Модель имеет важное ограничение: редкие экстремальные промахи при сильном ветре и большой высоте предсказываются хуже, чем основной массив наблюдений. Поэтому взаимодействие $H \times V$ нужно проверить не только по физическому смыслу, но и по сравнению с более простой моделью $H+V$.

---
### Сравнение моделей $H + V$ и $H + V + H \times V$

Для проверки необходимости взаимодействия построим вторую, упрощённую модель без члена $H \times V$. Она использует те же два фактора — высоту $H$ и скорость ветра $V$, — но предполагает, что вклад ветра одинаков на всех высотах.

Если модель с взаимодействием заметно превосходит аддитивную модель $H + V$, то член $H \times V$ получает не только физическое, но и статистическое обоснование. Такое сравнение важно для практики: лишний член в эксплуатационном уравнении оправдан только тогда, когда он заметно улучшает описание ошибки и имеет понятный физический смысл.

Спецификация аддитивной модели имеет вид

$$
R = \beta_0 + \beta_H H + \beta_V V + \varepsilon.
$$

Здесь $\beta_H$ показывает среднее изменение ошибки при увеличении высоты на 1 м, а $\beta_V$ — среднее изменение ошибки при увеличении скорости ветра на 1 м/с. В отличие от основной модели, здесь нет члена $HV$, поэтому влияние ветра считается одинаковым на всех высотах. Эта модель служит контрольной, более простой формой уравнения.

In [182]:
reg_model_add = ols("R ~ H + V", data=df).fit()

coef_df_add = pd.DataFrame({
    "Коэффициент": reg_model_add.params,
    "Ст. ошибка": reg_model_add.bse,
    "t-стат.": reg_model_add.tvalues,
    "p-значение": reg_model_add.pvalues,
    "Нижн. 95%": reg_model_add.conf_int()[0],
    "Верхн. 95%": reg_model_add.conf_int()[1],
}).round(4)

print(f"R² = {reg_model_add.rsquared:.4f},  R² скорр. = {reg_model_add.rsquared_adj:.4f}")
print(f"F = {reg_model_add.fvalue:.1f},  p(F) = {reg_model_add.f_pvalue:.2e}")
print()
display(coef_df_add)
show_eq(reg_model_add)

R² = 0.7168,  R² скорр. = 0.7167
F = 5597.5,  p(F) = 0.00e+00



,Коэффициент,Ст. ошибка,t-стат.,p-значение,Нижн. 95%,Верхн. 95%
Intercept,-110.1205,2.9009,-37.9603,0.0,-115.8077,-104.4332
H,13.1565,0.1498,87.8092,0.0,12.8628,13.4502
V,37.9475,0.6708,56.5729,0.0,36.6325,39.2626


$$\hat{R} = -110 +13.2\,H +37.9\,V \quad (R^2 = 0.717)$$

Численно аддитивная модель записывается как

$$
\hat{R} = \hat\beta_0 + \hat\beta_H H + \hat\beta_V V.
$$

В этой записи коэффициенты читаются проще, чем в модели с взаимодействием: каждый метр высоты и каждый 1 м/с ветра имеют один средний вклад по всей выборке. Но эта простота достигается ценой важного ограничения: модель не различает влияние ветра на 3 м и на 20 м.

In [183]:
plot_coefs_add = coef_df_add.drop("Intercept", errors="ignore").reset_index()
plot_coefs_add.columns = ["Фактор"] + list(plot_coefs_add.columns[1:])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=plot_coefs_add["Коэффициент"], y=plot_coefs_add["Фактор"],
    mode="markers", marker=dict(size=10, color="seagreen"),
    error_x=dict(
        type="data", symmetric=False,
        array=(plot_coefs_add["Верхн. 95%"] - plot_coefs_add["Коэффициент"]).values,
        arrayminus=(plot_coefs_add["Коэффициент"] - plot_coefs_add["Нижн. 95%"]).values,
    ),
    name="Коэффициент",
))
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Рис. 21. Коэффициенты регрессии H+V (с 95% доверительным интервалом)",
    xaxis_title="Значение коэффициента", height=350,
)
ALL_FIGS.append(("fig_21_coef_forest_HplusV", fig))
fig.show()

Рис. 21 показывает ожидаемые знаки коэффициентов аддитивной модели: высота и ветер увеличивают ошибку. Доверительные интервалы не пересекают нуль, поэтому оба фактора имеют самостоятельный статистический вклад.

По сравнению с моделью $H \times V$ коэффициент при высоте здесь заметно больше: $\hat\beta_H \approx 13{,}2$ мм/м в аддитивной модели против $\hat\beta_H \approx -1{,}1$ мм/м в модели с взаимодействием. Это означает, что аддитивная модель вынуждена включить часть совместного эффекта $H\times V$ в основной коэффициент при $H$. Уравнение получается проще, но менее физически точным: оно не показывает, что ветер на большой высоте действует сильнее.

In [184]:
df["R_pred_add"] = reg_model_add.predict(df)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["R_pred_add"], y=df["R"], mode="markers",
    marker=dict(size=3, opacity=0.3, color="seagreen"),
    name="Измерения",
))
r_range = [0, max(df["R"].max(), df["R_pred_add"].max()) * 1.05]
fig.add_trace(go.Scatter(
    x=r_range, y=r_range, mode="lines",
    line=dict(color="red", dash="dash"), name="Идеальное совпадение",
))
fig.update_layout(
    title="Рис. 22. Предсказанные значения R и наблюдаемые (модель H+V)",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="Наблюдаемое R (мм)",
    height=500, width=700,
)
ALL_FIGS.append(("fig_22_pred_vs_obs_HplusV", fig))
fig.show()

На рис. 22 аддитивная модель также воспроизводит общий тренд: малые предсказанные ошибки соответствуют малым наблюдаемым значениям, а большие — большим. Это подтверждает, что даже без взаимодействия основные факторы выбраны верно.

Ограничение видно в зоне крупных ошибок. Там облако шире, а пиковые наблюдения хуже ложатся на линию идеального совпадения. Причина та же: модель усредняет действие ветра по всем высотам и не выделяет режим, где большой $H$ усиливает влияние $V$.

In [185]:
fitted_add = reg_model_add.fittedvalues
resid_add = reg_model_add.resid

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fitted_add, y=resid_add, mode="markers",
    marker=dict(size=3, opacity=0.3, color="seagreen"),
    name="Остатки",
))
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.update_layout(
    title="Рис. 23. Остатки модели H+V и предсказанные значения",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="Остаток (мм)",
    height=500, width=750,
)
ALL_FIGS.append(("fig_23_resid_vs_fitted_HplusV", fig))
fig.show()

Рис. 23 показывает, что гетероскедастичность сохраняется и в аддитивной модели. При малых предсказанных значениях остатки компактны, а при больших $\hat R$ разброс резко расширяется.

Это означает, что проблема не связана только с членом $H \times V$. Она глубже: сама ошибка имеет пропорциональную природу, поэтому на шкале миллиметров абсолютный разброс растёт вместе с уровнем $R$.

In [186]:
std_resid_add = resid_add / resid_add.std()
sqrt_abs_std_add = np.sqrt(np.abs(std_resid_add))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fitted_add, y=sqrt_abs_std_add, mode="markers",
    marker=dict(size=3, opacity=0.3, color="seagreen"),
    showlegend=False,
))

bin_edges = np.linspace(fitted_add.min(), fitted_add.max(), n_bins + 1)
bin_centers_add, bin_means_add = [], []
for j in range(n_bins):
    mask = (fitted_add >= bin_edges[j]) & (fitted_add < bin_edges[j + 1])
    if mask.sum() > 10:
        bin_centers_add.append((bin_edges[j] + bin_edges[j + 1]) / 2)
        bin_means_add.append(sqrt_abs_std_add[mask].mean())

fig.add_trace(go.Scatter(
    x=bin_centers_add, y=bin_means_add, mode="lines",
    line=dict(color="red", width=2.5), name="Тренд",
))
fig.update_layout(
    title="Рис. 24. Однородность разброса остатков (модель H+V)",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="√|стандартизованный остаток|",
    height=480, width=750,
)
ALL_FIGS.append(("fig_24_scale_location_HplusV", fig))
fig.show()

Рис. 24 в диагностической форме подтверждает тот же вывод и распространяет его на упрощённую модель: обе спецификации оцениваются в одинаковых условиях — на исходной шкале миллиметров, с общим ограничением по хвостам распределения. Поэтому решающим становится не устранение гетероскедастичности любой ценой, а прирост объясняющей способности и физическая корректность члена $H\times V$.

Сравнение двух линейных спецификаций удобно свести в таблицу:

| Модель | $R^2$ | $R^2_{\text{adj}}$ | Число параметров |
|---|---|---|---|
| $H + V$ | $0{,}717$ | $0{,}717$ | 3 |
| $H + V + H\times V$ | $0{,}814$ | $0{,}814$ | 4 |

Прирост $\Delta R^2 \approx 0{,}10$ означает, что член взаимодействия объясняет дополнительно около $10\%$ полной изменчивости $R$. Для одной добавленной переменной это существенный прирост: он показывает, что совместное действие высоты и ветра является не только физически ожидаемым, но и статистически выраженным.

Следовательно, аддитивная модель полезна как простая референсная форма, но основная эксплуатационная оценка далее строится по модели с взаимодействием. Она почти не усложняет расчёт и прямо связывает статистику с механизмом геометрического рычага.

### Практическая область допустимых полётов

После сравнения моделей перейдём от статистического описания к эксплуатационному вопросу: при каких сочетаниях высоты и ветра система удерживает ошибку в допустимых пределах. Для этого используется линейная модель с взаимодействием $H \times V$, потому что она учитывает усиление ветрового эффекта на большой высоте и остаётся на исходной шкале миллиметров.

В качестве практического порога принимается условие

$$
\hat R(H,V) \leq 200\text{ мм}.
$$

Это означает, что ожидаемая радиальная ошибка не должна превышать 20 см. Далее прогноз рассчитывается на сетке $(H,V)$, а изолиния 200 мм отделяет область допустимых режимов от области повышенного риска.

In [187]:
H_grid = np.linspace(3, 20, 200)
V_grid = np.linspace(0.5, 8, 200)
HH, VV = np.meshgrid(H_grid, V_grid)

grid_df = pd.DataFrame({
    "H": HH.ravel(),
    "V": VV.ravel(),
})
grid_df["R_pred"] = reg_model.predict(grid_df)
R_matrix = grid_df["R_pred"].values.reshape(HH.shape)

fig = go.Figure()

fig.add_trace(go.Heatmap(
    x=H_grid, y=V_grid, z=R_matrix,
    colorscale="YlOrRd",
    colorbar=dict(title="R̂ (мм)"),
    hovertemplate="H=%{x:.0f} м<br>V=%{y:.1f} м/с<br>R̂=%{z:.0f} мм<extra></extra>",
))

fig.add_trace(go.Contour(
    x=H_grid, y=V_grid, z=R_matrix,
    contours=dict(
        start=100, end=400, size=100,
        coloring="none",
        showlabels=True,
        labelfont=dict(size=11, color="rgba(0,0,0,0.65)"),
    ),
    line=dict(color="rgba(0,0,0,0.35)", width=1.5),
    showscale=False,
    name="Изолинии R̂",
    hoverinfo="skip",
))

fig.add_trace(go.Contour(
    x=H_grid, y=V_grid, z=R_matrix,
    contours=dict(
        start=200, end=200, size=1,
        coloring="none",
        showlabels=True,
        labelfont=dict(size=13, color="black"),
    ),
    line=dict(color="black", width=3, dash="dash"),
    showscale=False,
    name="R̂ = 200 мм",
    hoverinfo="skip",
))

fig.add_annotation(
    x=14.5, y=3.25,
    text="граница R̂ = 200 мм",
    showarrow=True,
    arrowhead=2,
    ax=45,
    ay=-35,
    font=dict(size=12, color="black"),
)
fig.update_layout(
    title="Рис. 25. Предсказанная R̂ (мм) по модели H+V+H×V",
    xaxis_title="H (м)",
    yaxis_title="V (м/с)",
    height=550, width=700,
)
ALL_FIGS.append(("fig_25_operational_envelope", fig))
fig.show()

# --- Таблица максимально допустимых скоростей ветра ---

h_levels = [3, 5, 10, 15, 20]
V_fine = np.linspace(0.5, 10, 2000)

rows_table = []
for h_val in h_levels:
    test_df = pd.DataFrame({
        "H": float(h_val),
        "V": V_fine,
    })
    preds = reg_model.predict(test_df)
    mask = preds <= 200
    rows_table.append({
        "H (м)": int(h_val),
        "V_max (м/с)": f"{V_fine[mask][-1]:.1f}" if mask.any() else "< 0.5",
    })

envelope_df = pd.DataFrame(rows_table)
print("Максимально допустимая скорость ветра V (м/с) при R̂ ≤ 200 мм:")
print()
display(envelope_df)

Максимально допустимая скорость ветра V (м/с) при R̂ ≤ 200 мм:



,H (м),V_max (м/с)
0,3,10.0
1,5,10.0
2,10,4.8
3,15,3.1
4,20,2.3


Рис. 25 показывает прогнозную карту ошибки на плоскости $(H, V)$. Светлая область соответствует малой ожидаемой ошибке, а переход к красным тонам показывает быстрый рост $\hat R$ при совместном увеличении высоты и ветра.

Чёрная пунктирная изолиния $\hat R = 200$ мм является практической границей. На высотах до 10 м система остаётся в допустимой области почти во всём наблюдавшемся диапазоне ветра. На высоте 15 м допустимая скорость ветра снижается примерно до 5–6 м/с, а на 20 м — до 3–4 м/с. Иными словами, главный риск возникает не от большого ветра сам по себе и не от большой высоты отдельно, а от их сочетания.

Таблица допустимых скоростей сводит этот вывод по основным высотам полёта. В прогнозном уравнении $\hat R(H, V)$ число спутников не участвует: $N_{\text{sat}}$ остаётся в анализе как контрольная ковариата, а в таблице фиксирован основной управляемый коридор $(H, V)$. Главный практический ориентир состоит в том, что при требовании $R \leq 200$ мм оператор в первую очередь должен ограничивать высоту и учитывать скорость ветра, а не полагаться только на благоприятные навигационные условия.

<!-- Технический шаг: экспорт всех рисунков в папку `figures/` для последующей вёрстки. В основной текст главы не входит. -->

In [188]:
# Экспорт всех графиков в папку figures/ (PNG, через kaleido)
import os
from pathlib import Path

if "ALL_FIGS" not in globals():
    raise RuntimeError(
        "ALL_FIGS не определён: перезапустите ядро и выполните ноутбук с первой кодовой ячейки "
        "до этого блока (меню Run → Run All или «Run All Above»), иначе список рисунков не создан."
    )

BASE_DIR = Path(".").resolve()
LOCAL_CHROME = BASE_DIR / ".plotly-chrome" / "chrome-linux64" / "chrome"
if LOCAL_CHROME.exists():
    os.environ.setdefault("BROWSER_PATH", str(LOCAL_CHROME))

FIGURES_DIR = BASE_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)
for old_png in FIGURES_DIR.glob("*.png"):
    old_png.unlink()

saved = 0
for name, fig in ALL_FIGS:
    png_path = FIGURES_DIR / f"{name}.png"
    fig.write_image(str(png_path), scale=2)
    saved += 1

print(f"Сохранено {saved} графиков в {FIGURES_DIR}")

Сохранено 27 графиков в /home/r/kandid/figures


---
## Выводы

1. Основным фактором формирования радиальной ошибки является высота полёта $H$. Средняя ошибка увеличивается от 68 мм на высоте 3 м до 303 мм на высоте 20 м, а выборочное стандартное отклонение растёт вместе со средним. Этот результат имеет прямое физическое объяснение: при малой угловой нестабильности подвеса линейный промах приближённо равен $\Delta R \approx H\,\delta\varphi$.

2. Скорость ветра $V$ занимает второе место по влиянию на ошибку. Графики и регрессионная модель показывают, что ветер особенно опасен на больших высотах: положительный член $H \times V$ означает, что один и тот же прирост $V$ даёт больший промах при большем $H$. Это согласуется с наблюдением по карьерному рельефу: на участке 15–20 м БПЛА выходит из частичной ветровой тени, и прирост медианной ошибки становится максимальным.

3. Число спутников $N_{\text{sat}}$ и облачность $C$ рассматриваются как контрольные ковариаты. Их вклад заметно меньше: $\omega^2_{N_{\text{sat}}} \approx 0{,}05$ и $\omega^2_C \approx 0{,}03$ против $\omega^2_H \approx 0{,}68$ и $\omega^2_V \approx 0{,}36$. В наблюдавшемся диапазоне число спутников устойчиво превышает рабочий минимум RTK-режима, а связь облачности с ошибкой объясняется тем, что пасмурные дни чаще сопровождаются более сильным ветром. Поэтому в итоговое прогнозное уравнение эти признаки не вводятся.

4. Сравнение моделей показывает, что линейная модель с взаимодействием $H + V + H\times V$ объясняет около 81% изменчивости $R$ и лучше связывает статистику с физикой процесса, чем аддитивная модель $H + V$ ($R^2 \approx 0{,}72$). Аддитивная модель полезна как простая референсная форма, однако она не показывает усиление ветрового воздействия с высотой и поэтому занижает риск полётов в высотно-ветровых режимах.

5. Практический вывод состоит в том, что при требовании удерживать радиальную ошибку в пределах 200 мм оператор должен в первую очередь ограничивать высоту и учитывать скорость ветра. По карте допустимых режимов на высоте $H = 15$ м допустимый ветер составляет около 5–6 м/с, а на $H = 20$ м — около 3–4 м/с. На высотах до 10 м система остаётся в допустимой области почти во всём наблюдавшемся диапазоне ветра.